<a href="https://colab.research.google.com/github/felipeconradovidal/SCADA-Core_Automatica_GRUPO4/blob/main/etapa-01-logica/03%20-%20Notebook%20de%20tautologias%20e%20contradicoes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook: Variação lógica das tags e teste de tautologias/contradições
## Etapa 01 – Lógica | Parte 3 — Planta de Classificação de Grãos

Este notebook simula o **cenário real do projeto**: a planta automatizada de recepção, pesagem, inspeção por visão
computacional e ejeção pneumática de grãos, descrita em `00 - Descritivo Do Processo.md` e formalizada em
`01 - Variáveis do Processo.md` e `02 - Lógica Proposicional.md`.

O objetivo é gerar variações lógicas das proposições do processo e
verificar, para cada expressão relevante do intertravamento, da classificação e da ejeção, se ela é:

- **Tautologia**: sempre verdadeira, para toda combinação das variáveis envolvidas;
- **Contradição**: sempre falsa;
- **Contingente**: depende da combinação de entradas (comportamento esperado da maior parte das regras de processo).

### Redução do espaço de estados por relevância
Como o cenário do projeto reúne cerca de 24 proposições atômicas, testar exaustivamente **todas** as
variáveis contra **todas** as expressões exigiria 2²⁴ ≈ 16,7 milhões de combinações — algo lento de
processar e pouco didático de acompanhar, além de desnecessário, já que a maioria das expressões depende
apenas de um subconjunto pequeno das variáveis. Por isso, cada expressão declara explicitamente **quais
variáveis lhe são relevantes**, e a checagem exaustiva é feita apenas sobre esse subconjunto — o que é
logicamente equivalente (as demais variáveis não têm influência no resultado) e muito mais rápido de
simular.

In [ ]:
import itertools
import random
import pandas as pd

# ---------------------------------------------------------------------------
# 1. Proposições atômicas do processo (entradas independentes da lógica)
# ---------------------------------------------------------------------------
# Cada uma corresponde a um símbolo lógico da tabela de variáveis (Parte 2 do
# documento "01 - Variáveis do Processo.md") ou a um comparador de limiar
# derivado de uma variável analógica (Seção 2 de "02 - Lógica Proposicional.md").

variaveis = [
    # Segurança / intertravamento geral
    'p_EMERG',    # XA-901  - botoeira de emergência
    'p_JI201',    # JI-201  - sobrecarga no motor da esteira
    'p_PAL601',   # PAL-601 - pressão pneumática baixa
    'p_KSA401',   # KSA-401 - status do sistema de visão/câmera
    # Nível do funil de recepção (comparadores de LIT-101)
    'p_NB101',    # nível baixo
    'p_NA101',    # nível alto
    'p_NC101',    # nível crítico
    # Velocidade da esteira (comparadores de ST-201)
    'p_MOV201',   # esteira em movimento
    'p_VB201',    # velocidade abaixo da faixa
    'p_VA201',    # velocidade acima da faixa
    # Reservatório de rejeito (comparadores de LIT-703)
    'p_NA703',    # reservatório cheio (alarme ~90%)
    'p_NC703',    # nível crítico (bloqueio 100%)
    # Estação de visão / ejeção
    'p_XS401',    # XS-401  - trigger fotoelétrico
    'p_ZSH601',   # ZSH-601 - confirmação física do atuador pneumático
    'p_POS603',   # posição calculada do grão no bocal de ejeção (shift register)
    # Visão computacional (Tabela 1 - CV-101 a CV-109)
    'p_CV101', 'p_CV102',   # cor ideal / secundária
    'p_CV103', 'p_CV104',   # tamanho ideal / secundário
    'p_CV105', 'p_CV106',   # formato ideal / secundário
    'p_CV107',               # dano
    'p_CV108',               # praga
    'p_CV109',               # impureza
]

print(f'Total de proposições atômicas mapeadas: {len(variaveis)}')
print(f'Espaço de estados completo (2^{len(variaveis)}): {2**len(variaveis):,}'.replace(',', '.'))

def gerar_estado_aleatorio():
    """Sorteia um estado lógico aleatório para todas as proposições atômicas."""
    return {v: random.choice([True, False]) for v in variaveis}

print('\nExemplo de estado aleatório:')
print(gerar_estado_aleatorio())

## 2. Regras do processo (Seção 3 a 7 de `02 - Lógica Proposicional.md`)

As funções abaixo implementam, em Python, cada expressão lógica formal do documento de referência do
projeto. Todas recebem um dicionário de estado (`vars_`) e retornam um booleano.

Foi adotada a versão **consolidada** de `c_PERM` (Seção 7), que já incorpora o bloqueio por reservatório de
rejeito cheio (`¬p_NC703`), substituindo a versão inicial da Seção 3.

In [4]:
# --- Proposição derivada da velocidade da esteira (Seção 2.2) -------------
def p_VN201(vars_):
    """Velocidade normal: nem abaixo, nem acima da faixa, e esteira em movimento."""
    return (not vars_['p_VB201']) and (not vars_['p_VA201']) and vars_['p_MOV201']

# --- Permissão Geral de Operação (Seção 3 + consolidação da Seção 7) ------
def c_PERM(vars_):
    return (
        (not vars_['p_EMERG'])
        and (not vars_['p_JI201'])
        and (not vars_['p_PAL601'])
        and vars_['p_KSA401']
        and (not vars_['p_NC703'])
    )

# --- Comando do Alimentador Vibratório (Seção 4) ---------------------------
def c_ALIM(vars_):
    return c_PERM(vars_) and vars_['p_MOV201'] and (not vars_['p_NB101'])

# --- Classificação dos grãos (Seção 5) -------------------------------------
def p_A(vars_):
    """Categoria A: aprovado integralmente."""
    return (
        vars_['p_CV101'] and vars_['p_CV103'] and vars_['p_CV105']
        and not vars_['p_CV107'] and not vars_['p_CV108'] and not vars_['p_CV109']
    )

def p_C(vars_):
    """Categoria C: rejeitado (defeito grave ou totalmente fora do padrão)."""
    return (
        vars_['p_CV107'] or vars_['p_CV108'] or vars_['p_CV109']
        or (not vars_['p_CV101'] and not vars_['p_CV102'])
        or (not vars_['p_CV103'] and not vars_['p_CV104'])
        or (not vars_['p_CV105'] and not vars_['p_CV106'])
    )

def p_B(vars_):
    """Categoria B: definida por exclusão (nem A, nem C)."""
    return (not p_A(vars_)) and (not p_C(vars_))

# --- Sistema de ejeção pneumática (Seção 6) --------------------------------
def c_FY603(vars_):
    return p_C(vars_) and vars_['p_POS603'] and (not vars_['p_PAL601'])

def p_FALHA_EJETOR(vars_):
    return c_FY603(vars_) and (not vars_['p_ZSH601'])

# --- Alarmes de processo (Seção 7) -----------------------------------------
def Alarme_JI201(vars_):
    return vars_['p_JI201']

def Alarme_PAL601(vars_):
    return vars_['p_PAL601']

def Alarme_LIT703(vars_):
    return vars_['p_NA703']

def Bloqueio_LIT703(vars_):
    return vars_['p_NC703']

def Alarme_EJETOR(vars_):
    return p_FALHA_EJETOR(vars_)

## 3. Classificação das expressões (tautologia / contradição / contingente)

Cada expressão a testar é registrada com:

- a função que a implementa;
- a lista de proposições **relevantes** para ela (usadas na checagem exaustiva);
- uma leitura em português do que está sendo verificado.

A checagem exaustiva usa `itertools.product` apenas sobre as variáveis relevantes — as demais são fixadas
em `False` por padrão, já que não afetam o resultado da expressão.

In [ ]:
def avaliar_exaustivo(fn, vars_relevantes):
    """Avalia fn sobre todas as combinações das vars_relevantes (demais = False)."""
    resultados = []
    for combinacao in itertools.product([False, True], repeat=len(vars_relevantes)):
        estado = {v: False for v in variaveis}
        estado.update(dict(zip(vars_relevantes, combinacao)))
        resultados.append(fn(estado))
    return resultados

def classificar(valores):
    if all(valores):
        return 'Tautologia'
    if not any(valores):
        return 'Contradição'
    return 'Contingente'

# Cada entrada: (nome, função, variáveis relevantes, leitura)
expressoes = [
    (
        'c_PERM — Permissão Geral de Operação',
        c_PERM,
        ['p_EMERG', 'p_JI201', 'p_PAL601', 'p_KSA401', 'p_NC703'],
        'Contingente por natureza: depende das 5 condições de segurança monitoradas.',
    ),
    (
        'c_ALIM — Comando do Alimentador Vibratório',
        c_ALIM,
        ['p_EMERG', 'p_JI201', 'p_PAL601', 'p_KSA401', 'p_NC703', 'p_MOV201', 'p_NB101'],
        'Contingente: só liga com planta liberada, esteira em movimento e funil sem nível baixo.',
    ),
    (
        'Exaustividade da partição: p_A ∨ p_B ∨ p_C',
        lambda v: p_A(v) or p_B(v) or p_C(v),
        ['p_CV101', 'p_CV102', 'p_CV103', 'p_CV104', 'p_CV105', 'p_CV106',
         'p_CV107', 'p_CV108', 'p_CV109'],
        'Deve ser TAUTOLOGIA: todo grão cai em pelo menos uma categoria (A, B ou C), por construção de p_B.',
    ),
    (
        'Exclusividade A ∧ C — ¬(p_A ∧ p_C)',
        lambda v: not (p_A(v) and p_C(v)),
        ['p_CV101', 'p_CV102', 'p_CV103', 'p_CV104', 'p_CV105', 'p_CV106',
         'p_CV107', 'p_CV108', 'p_CV109'],
        'Deve ser TAUTOLOGIA: nenhum grão pode ser Categoria A e C ao mesmo tempo.',
    ),
    (
        'Exclusividade A ∧ B — ¬(p_A ∧ p_B)',
        lambda v: not (p_A(v) and p_B(v)),
        ['p_CV101', 'p_CV102', 'p_CV103', 'p_CV104', 'p_CV105', 'p_CV106',
         'p_CV107', 'p_CV108', 'p_CV109'],
        'TAUTOLOGIA trivial, já que p_B é definido como ¬p_A ∧ ¬p_C.',
    ),
    (
        'Exclusividade B ∧ C — ¬(p_B ∧ p_C)',
        lambda v: not (p_B(v) and p_C(v)),
        ['p_CV101', 'p_CV102', 'p_CV103', 'p_CV104', 'p_CV105', 'p_CV106',
         'p_CV107', 'p_CV108', 'p_CV109'],
        'TAUTOLOGIA trivial, pela mesma razão da anterior.',
    ),
    (
        'Propagação do bloqueio: c_ALIM → ¬p_NC703',
        lambda v: (not c_ALIM(v)) or (not v['p_NC703']),
        ['p_EMERG', 'p_JI201', 'p_PAL601', 'p_KSA401', 'p_NC703', 'p_MOV201', 'p_NB101'],
        'Deve ser TAUTOLOGIA: se o reservatório de rejeito está em nível crítico, o alimentador nunca pode estar ligado.',
    ),
    (
        'c_FY603 → p_C (só ejeta grão de Categoria C)',
        lambda v: (not c_FY603(v)) or v['p_C_fixture'],
        [],  # tratada abaixo com fixture dedicada
        'Deve ser TAUTOLOGIA: a válvula nunca abre para um grão que não seja Categoria C.',
    ),
    (
        'c_FY603 → ¬p_PAL601 (não ejeta sem pressão OK)',
        lambda v: (not c_FY603(v)) or (not v['p_PAL601']),
        ['p_CV101', 'p_CV102', 'p_CV103', 'p_CV104', 'p_CV105', 'p_CV106',
         'p_CV107', 'p_CV108', 'p_CV109', 'p_POS603', 'p_PAL601'],
        'Deve ser TAUTOLOGIA: o comando de ejeção exige explicitamente ¬p_PAL601.',
    ),
    (
        'p_FALHA_EJETOR → c_FY603',
        lambda v: (not p_FALHA_EJETOR(v)) or c_FY603(v),
        ['p_CV101', 'p_CV102', 'p_CV103', 'p_CV104', 'p_CV105', 'p_CV106',
         'p_CV107', 'p_CV108', 'p_CV109', 'p_POS603', 'p_PAL601', 'p_ZSH601'],
        'Deve ser TAUTOLOGIA: só existe falha de ejeção se houve, antes, um comando de ejeção.',
    ),
    (
        'c_PERM → ¬p_EMERG (segurança básica)',
        lambda v: (not c_PERM(v)) or (not v['p_EMERG']),
        ['p_EMERG', 'p_JI201', 'p_PAL601', 'p_KSA401', 'p_NC703'],
        'Deve ser TAUTOLOGIA: a planta jamais é liberada em estado de emergência.',
    ),
    (
        'Contradição clássica: p_EMERG ∧ ¬p_EMERG',
        lambda v: v['p_EMERG'] and not v['p_EMERG'],
        ['p_EMERG'],
        'Referência clássica: sempre falsa, por definição de negação.',
    ),
    (
        'Tautologia clássica: p_EMERG ∨ ¬p_EMERG',
        lambda v: v['p_EMERG'] or not v['p_EMERG'],
        ['p_EMERG'],
        'Referência clássica: sempre verdadeira, terceiro excluído.',
    ),
]

# A expressão "c_FY603 → p_C" precisa de p_C calculado a partir das mesmas CV,
# então construímos manualmente (sem lambda de fixture fictícia usada acima).
def _c_fy603_implica_pC(vars_):
    return (not c_FY603(vars_)) or p_C(vars_)

expressoes[7] = (
    'c_FY603 → p_C (só ejeta grão de Categoria C)',
    _c_fy603_implica_pC,
    ['p_CV101', 'p_CV102', 'p_CV103', 'p_CV104', 'p_CV105', 'p_CV106',
     'p_CV107', 'p_CV108', 'p_CV109', 'p_POS603', 'p_PAL601'],
    'Deve ser TAUTOLOGIA: a válvula nunca abre para um grão que não seja Categoria C.',
)

resultado = []
for nome, fn, vars_rel, leitura in expressoes:
    valores = avaliar_exaustivo(fn, vars_rel)
    resultado.append({
        'Expressão': nome,
        'Nº vars relevantes': len(vars_rel),
        'Combinações testadas': len(valores),
        'Verdadeiras': sum(valores),
        'Falsas': len(valores) - sum(valores),
        'Classificação': classificar(valores),
    })

df_resultado = pd.DataFrame(resultado)
print(df_resultado.to_string(index=False))

### Leitura dos resultados

Para consultar a leitura textual de cada expressão junto da classificação obtida:

In [ ]:
for (nome, fn, vars_rel, leitura), linha in zip(expressoes, resultado):
    print(f"[{linha['Classificação']:^11}] {nome}")
    print(f"    -> {leitura}\n")

## 4. Simulação aleatória de estados do processo

Diferente do notebook de referência, aqui exibimos, para cada estado sorteado, **todo o encadeamento**
descrito na Seção 8 de `02 - Lógica Proposicional.md`: da permissão geral até o diagnóstico de falha de
ejeção, passando pela classificação do grão.

In [ ]:
def resumo_estado(vars_):
    return {
        'c_PERM': c_PERM(vars_),
        'c_ALIM': c_ALIM(vars_),
        'p_A (Categoria A)': p_A(vars_),
        'p_B (Categoria B)': p_B(vars_),
        'p_C (Categoria C)': p_C(vars_),
        'c_FY603 (aciona ejetor)': c_FY603(vars_),
        'p_FALHA_EJETOR': p_FALHA_EJETOR(vars_),
        'Alarme_JI201': Alarme_JI201(vars_),
        'Alarme_PAL601': Alarme_PAL601(vars_),
        'Alarme_LIT703': Alarme_LIT703(vars_),
        'Bloqueio_LIT703': Bloqueio_LIT703(vars_),
    }

for i in range(5):
    estado = gerar_estado_aleatorio()
    print(f'\n=== Estado #{i + 1} ===')
    entradas_chave = {k: estado[k] for k in (
        'p_EMERG', 'p_JI201', 'p_PAL601', 'p_KSA401', 'p_NC703',
        'p_MOV201', 'p_NB101', 'p_POS603', 'p_ZSH601',
    )}
    print('Entradas-chave:', entradas_chave)
    for nome, valor in resumo_estado(estado).items():
        print(f'  {nome:<26} = {valor}')

## 5. Restrições físicas (fora do escopo da álgebra booleana pura)

A álgebra proposicional, por si só, não impede combinações **logicamente possíveis mas fisicamente
impossíveis** — por exemplo, `p_NB101` (nível baixo) e `p_NA101` (nível alto) verdadeiros ao mesmo tempo,
já que ambos são comparadores independentes de um único sinal analógico (`LIT-101`). O mesmo vale para
`p_VB201`/`p_VA201` (velocidade abaixo/acima da faixa).

Essas restrições precisam ser impostas separadamente — seja como pré-condição na geração de estados de
teste, seja como verificação de consistência no CLP. A função abaixo sinaliza estados logicamente válidos
mas fisicamente inconsistentes.

In [ ]:
def estado_fisicamente_valido(vars_):
    """Retorna False se o estado viola restrições físicas dos comparadores analógicos."""
    violacoes = []
    if vars_['p_NB101'] and vars_['p_NA101']:
        violacoes.append('LIT-101 não pode estar simultaneamente em nível baixo e nível alto')
    if vars_['p_VB201'] and vars_['p_VA201']:
        violacoes.append('ST-201 não pode estar simultaneamente abaixo e acima da faixa')
    if vars_['p_NA703'] and vars_['p_NC703'] is False and vars_['p_NC703']:
        pass  # placeholder para futuras regras (p_NC703 >= p_NA703 fisicamente, ver nota abaixo)
    return (len(violacoes) == 0), violacoes

print('Verificando os 5 estados simulados acima quanto à consistência física...\n')
random.seed(42)
for i in range(5):
    estado = gerar_estado_aleatorio()
    valido, violacoes = estado_fisicamente_valido(estado)
    status = 'OK' if valido else 'INCONSISTENTE'
    print(f'Estado #{i + 1}: {status}')
    for v in violacoes:
        print(f'   - {v}')

print(
    '\nNota: como LIT-703 cresce de 0% a 100%, fisicamente p_NC703 (>=100%) implica p_NA703 (>90%). '
    'Essa dependência pode ser adicionada como regra adicional (p_NC703 -> p_NA703) em uma próxima revisão.'
)

## 6. Interpretação

1. As provas de **exaustividade** e **exclusividade mútua** da partição A/B/C (linhas 3 a 6 da tabela)
   confirmam formalmente o que o documento `02 - Lógica Proposicional.md` já afirmava "por construção":
   todo grão cai em exatamente uma categoria.
2. A **propagação do bloqueio** (`c_ALIM → ¬p_NC703`) confirma que a consolidação de `c_PERM` na Seção 7
   realmente elimina a necessidade de uma lógica de bloqueio paralela — o intertravamento principal já
   cobre o caso do reservatório de rejeito cheio.
3. As regras de **segurança do ejetor** (`c_FY603 → p_C`, `c_FY603 → ¬p_PAL601`,
   `p_FALHA_EJETOR → c_FY603`) são tautologias por construção da fórmula — o que é o comportamento
   esperado e desejado para regras de intertravamento: elas devem ser *garantias*, não *coincidências*.
4. `c_PERM` e `c_ALIM` continuam **contingentes**, como esperado: são justamente as regras que devem
   variar conforme o estado real do processo, e não condições absolutas.
5. A checagem de **restrições físicas** (Seção 5) evidencia uma lacuna que a lógica proposicional pura não
   cobre sozinha: comparadores derivados do mesmo sinal analógico têm dependências entre si que precisam
   ser tratadas à parte (ou incorporadas como axiomas adicionais do sistema).
